# 💻 Notebook do Aluno — Aula 07: RAG avançado — chunking estratégico, reranking e RAGAS

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 07/14 — Módulo 2: RAG · 🏁 Entrega CKP02**  
**⏱️ 1h40min**  
**📊 faithfulness · answer_relevancy**  
**🏁 CKP02 entrega**  

---

## 🎯 Objetivo da aula

Sair da fase "funciona" para a fase "funciona bem". Medir objetivamente a qualidade do RAG com RAGAS, experimentar duas estratégias de chunking e documentar qual configuração entrega melhores resultados para o domínio do grupo. Isso é o CKP02.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime da aula.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-community langchain-ollama pymupdf chromadb ragas datasets langchain-experimental langchain-text-splitters -q

# Setup (idêntico à Aula 06)
import os
from google.colab import userdata, files
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
# 👉 LACUNA 1: carregue os PDFs do grupo (mesmos da Aula 06)
paginas = []
for pdf in pdf_paths:
    paginas.extend(PyMuPDFLoader(___).load())

# ── ESTRATÉGIA A: Recursive com chunk_size=512 (vs. 800 da Aula 06) ──
chunks_a = RecursiveCharacterTextSplitter(
    chunk_size=___, chunk_overlap=___
).split_documents(paginas)
db_a         = Chroma.from_documents(chunks_a, embeddings, persist_directory="/content/ckp02_a")
retriever_a  = db_a.as_retriever(search_kwargs={"k":3})
chain_a      = montar_chain_rag(retriever_a)

# ── ESTRATÉGIA B: SemanticChunker ────────────────────────────────────
# 👉 LACUNA 2: instancie o SemanticChunker com o modelo de embedding
chunks_b    = SemanticChunker(___).split_documents(paginas)
db_b        = Chroma.from_documents(chunks_b, embeddings, persist_directory="/content/ckp02_b")
retriever_b = ___  # retriever com k=3
chain_b     = montar_chain_rag(retriever_b)

# 👉 LACUNA 3: defina 5 perguntas reais do domínio para avaliação
PERGUNTAS = [___, ___, ___, ___, ___]

# 👉 LACUNA 4: monte o dataset RAGAS e avalie as duas estratégias
for nome, chain, retr in [
    ("Recursive-512", chain_a, retriever_a),
    ("Semantic",      chain_b, retriever_b),
]:
    dados = {
        "question": PERGUNTAS,
        "answer":   [chain.invoke(q) for q in PERGUNTAS],
        "contexts": [[d.page_content for d in retr.invoke(q)] for q in ___],
    }
    res = evaluate(Dataset.from_dict(dados), metrics=[faithfulness, answer_relevancy])
    print(f"{nome}: faith={res['faithfulness']:.3f} · rel={res['answer_relevancy']:.3f}")

---

## ✍️ Suas anotações

Registre aqui suas observações sobre o andaime e os exercícios (qualidade dos resultados, comparações e conclusões).

---

## 🏋️ Exercícios da Aula 07

Quatro exercícios práticos de otimização e avaliação do RAG — o chunking semântico, o reranking com cross-encoder, a avaliação com RAGAS e o diagnóstico do pior `faithfulness`.

Rode no Google Colab, na ordem, logo depois do andaime: a correção proposta no diagnóstico (trecho PRESENTE ou AUSENTE) é insumo direto do CKP02.


### Exercício 1 — Chunking semântico · ★★☆ · 10 min

*Individual · Colab*

1. Complete o tipo de quebra: `breakpoint_threshold_type=___` (o corte por percentil).
2. Complete o valor do corte: `breakpoint_threshold_amount=___` (95 é o padrão da aula).
3. Complete a lista passada ao `split_documents(___)` e compare com o Recursive-512.
4. Abra um chunk e verifique: o corte respeitou a fronteira de sentido?

> **💡 Dica:** o SemanticChunker usa o próprio modelo de embedding para decidir onde quebrar — o corte cai nos picos de divergência semântica.


In [ ]:
# Exercício 1 — chunking semântico (roda depois do lab: usa `paginas`)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

chunks_rec = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=64).split_documents(paginas)
print(f"Recursive-512: {len(chunks_rec)} chunks")

# 👉 LACUNA 1 e 2: o SemanticChunker decide onde quebrar com o próprio embedding
semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type=___,
    breakpoint_threshold_amount=___,
)
# 👉 LACUNA 3: a lista de páginas a dividir
chunks_sem = semantic_splitter.split_documents(___)
print(f"SemanticChunker: {len(chunks_sem)} chunks · média {sum(len(c.page_content) for c in chunks_sem)/len(chunks_sem):.0f} chars")
print(f"\nExemplo de chunk semântico:\n{chunks_sem[0].page_content[:200]}")
# 👉 LACUNA 4: o corte respeitou a fronteira de sentido? Compare com um chunk do Recursive.


### Exercício 2 — Reranking com cross-encoder · ★★☆ · 10 min

*Individual · Colab*

1. Complete o reranker: `CrossEncoderReranker(model=cross_encoder, top_n=___)` com top_n=3.
2. Complete o compressor e a busca ampla (`k=10`) no retriever com reranking.
3. Liste 3 perguntas reais do domínio.
4. Compare as ordens dos dois retrievers e anote o que mudou no pódio.

> **💡 Dica:** o bi-encoder compara query e documento como vetores separados; o cross-encoder lê o par (query, chunk) completo — mais preciso, por isso só no top-10.


In [ ]:
# Exercício 2 — reranking: bi-encoder vs. cross-encoder (roda depois do lab: usa `db_a`)
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker

cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

# 👉 LACUNA 1: crie o CrossEncoderReranker com o cross_encoder e top_n=3
reranker = ___

retriever_simples = db_a.as_retriever(search_kwargs={"k": 3})
retriever_rerank  = ContextualCompressionRetriever(
    base_compressor=___,                                         # 👉 LACUNA 2
    base_retriever=db_a.as_retriever(search_kwargs={"k": ___}),  # 👉 LACUNA 3: busca ampla (k=10)
)

# 👉 LACUNA 4: 3 perguntas reais do domínio
for q in [___, ___, ___]:
    print(f"\nQuery: {q}")
    print("── SEM reranking (distância cosseno):")
    for d in retriever_simples.invoke(q):
        print(f"   {d.page_content[:70]}")
    print("── COM reranking (cross-encoder reordena):")
    # 👉 LACUNA 5: compare com o retriever_rerank
    for d in ___:
        print(f"   {d.page_content[:70]}")


### Exercício 3 — Avaliação com RAGAS · ★★☆ · 10 min

*Individual · Colab*

1. Complete o juiz do RAGAS: o modelo de chat e a temperatura de avaliação.
2. Complete as duas métricas do Slide 12.
3. Rode a avaliação na estratégia A (Recursive-512) e leia a tabela.
4. Anote qual pergunta teve o pior `faithfulness` — ela vai para o Exercício 4.

> **💡 Dica:** o RAGAS usa o próprio Ollama como juiz (sem custo) — e a avaliação também roda com `temperature=0`.


In [ ]:
# Exercício 3 — avaliação RAGAS na estratégia A (roda depois do lab)
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset

# 👉 LACUNA 1 e 2: o juiz do RAGAS — o modelo de chat e a temperatura
ragas_llm = LangchainLLMWrapper(ChatOllama(model=___, temperature=___))
faithfulness.llm            = ragas_llm
answer_relevancy.llm        = ragas_llm
answer_relevancy.embeddings = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text"))

dados = {
    "question": PERGUNTAS,
    "answer":   [chain_a.invoke(q) for q in PERGUNTAS],
    "contexts": [[d.page_content for d in retriever_a.invoke(q)] for q in PERGUNTAS],
}
# 👉 LACUNA 3: as duas métricas do Slide 12
res = evaluate(Dataset.from_dict(dados), metrics=[___, ___])
print(res.to_pandas()[["question", "faithfulness", "answer_relevancy"]].sort_values("faithfulness"))


### Exercício 4 — Diagnóstico de faithfulness · ★★☆ · 10 min

*Individual · Colab*

1. Complete os campos do dataset de avaliação: perguntas e respostas.
2. Complete a ordenação da tabela por `faithfulness` (do pior para o melhor).
3. Recupere os chunks da pergunta crítica e leia os trechos.
4. Classifique a falha: trecho PRESENTE no contexto (prompt/LLM) ou AUSENTE (retriever)?

> **💡 Dica:** trecho presente + resposta errada = problema no prompt/LLM; trecho ausente = ajuste chunk_size/overlap ou k. Essa correção é insumo direto do CKP02.


In [ ]:
# Exercício 4 — diagnóstico de faithfulness (roda depois do Exercício 3)
dados_diag = {
    "question": ___,                              # 👉 LACUNA 1: as perguntas do lab
    "answer":   [chain_a.invoke(q) for q in ___], # 👉 LACUNA 2: as perguntas no loop
    "contexts": [[d.page_content for d in retriever_a.invoke(q)] for q in PERGUNTAS],
}
res_diag = evaluate(Dataset.from_dict(dados_diag), metrics=[faithfulness, answer_relevancy])
df = res_diag.to_pandas()

# 👉 LACUNA 3: ordene pela coluna de faithfulness (do pior para o melhor)
print(df[["question", "faithfulness"]].sort_values(by=___))

pior_q = df.sort_values("faithfulness").iloc[0]["question"]
print(f"\nPergunta crítica: {pior_q}\n")
# 👉 LACUNA 4: recupere os chunks da pergunta crítica
for d in ___.invoke(pior_q):
    print(d.page_content[:150])

# 👉 LACUNA 5: o trecho que responde a pergunta está nos chunks?
#   PRESENTE → problema no prompt/LLM: reforçar grounding, temperature=0
#   AUSENTE  → problema no retriever: ajustar chunk_size/overlap ou k


## 📚 Referências da aula

- Paper Es, S. et al. — "RAGAS: Automated Evaluation of Retrieval Augmented Generation." EACL, 2024. O paper que define faithfulness e answer_relevancy. arxiv.org/abs/2309.15217
- Docs RAGAS — Documentação oficial: métricas, integração com Ollama, datasets. docs.ragas.io
- Docs LangChain — SemanticChunker e ParentDocumentRetriever. python.langchain.com/docs/how_to/semantic-chunker
- Modelo cross-encoder/ms-marco-MiniLM-L-6-v2 — Modelo de reranking leve (22M params). huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2
- Livro Goodfellow, I.; Bengio, Y.; Courville, A. — Deep Learning. Pearson, 2017. Cap. 15 — Representações distribuídas: a base teórica dos embeddings que sustentam o chunking semântico.

---

**→ Próxima Aula — Aula 08 · 29/Set** — Interfaces com Gradio e Streamlit
  
RAG com URL pública. RunnableWithMessageHistory para memória entre turnos.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*